In [1]:
import os
import sys
# sys.path.insert(1, '')
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)

import vit_3d
import climate_dataset
import vivit
data_path = '/mnt/home/hwalters/data'
import numpy as np
import matplotlib.pyplot as plt

Current Working Directory: /Users/hayleywalters/Desktop/research 2025/ML_rain_prediction


In [2]:
import torch
import numpy as np

def download_and_prepare_dataset(data_path):
    """Utility function to download the dataset and return PyTorch tensors.

    Arguments:
        data_path (string): Path to the dataset
    """
    with np.load(data_path, allow_pickle=True) as data:
        # Get videos
        train_videos = np.nan_to_num(data["train_images"])
        valid_videos = np.nan_to_num(data["val_images"])
        test_videos = np.nan_to_num(data["test_images"])

        # Get labels
        train_labels = data["train_labels"]
        valid_labels = data["val_labels"]
        test_labels = data["test_labels"]

    # Convert to PyTorch tensors
    train_videos = torch.tensor(train_videos, dtype=torch.float32)
    valid_videos = torch.tensor(valid_videos, dtype=torch.float32)
    test_videos = torch.tensor(test_videos, dtype=torch.float32)
    train_labels = torch.tensor(train_labels, dtype=torch.long)
    valid_labels = torch.tensor(valid_labels, dtype=torch.long)
    test_labels = torch.tensor(test_labels, dtype=torch.long)

    return (
        (train_videos, train_labels),
        (valid_videos, valid_labels),
        (test_videos, test_labels),
    )

# Get the dataset
prepared_dataset = download_and_prepare_dataset("data/cluster_1_sst.npz")
(train_videos, train_labels) = prepared_dataset[0]
(valid_videos, valid_labels) = prepared_dataset[1]
(test_videos, test_labels) = prepared_dataset[2]

train_videos = train_videos[:, None, :, :, :]  # Add channel dim
valid_videos = valid_videos[:, None, :, :, :]
test_videos = test_videos[:, None, :, :, :]

print(f'train_videos {train_videos.shape}, train_labels {train_labels.shape}')
print(f'valid_videos {valid_videos.shape}, valid_labels {valid_labels.shape}')
print(f'test_videos {test_videos.shape}, test_labels {test_labels.shape}')

# def download_and_prepare_dataset(data_path):
#     """Utility function to download the dataset.

#     Arguments:
#         data_path (string): Path to the dataset
#     """
    
#     with np.load(data_path, allow_pickle=True) as data:
#         # Get videos
#         train_videos = np.nan_to_num(data["train_images"])
#         valid_videos = np.nan_to_num(data["val_images"])
#         test_videos = np.nan_to_num(data["test_images"])

#         # Get labels
#         train_labels = data["train_labels"]#.flatten()
#         valid_labels = data["val_labels"]#.flatten()
#         test_labels = data["test_labels"]#.flatten()

#     return (
#         (train_videos, train_labels),
#         (valid_videos, valid_labels),
#         (test_videos, test_labels),
#     )

# # Get the dataset
# prepared_dataset = download_and_prepare_dataset("data/cluster_1_sst.npz")
# (train_videos, train_labels) = prepared_dataset[0]
# (valid_videos, valid_labels) = prepared_dataset[1]
# (test_videos, test_labels) = prepared_dataset[2]
# train_videos = train_videos[:, None, :, :, :]  # Add channel dim
# valid_videos = valid_videos[:, None, :, :, :]
# test_videos = test_videos[:, None, :, :, :]


# print(f'train_videos {train_videos.shape}, train_labels {train_labels.shape}')
# print(f'valid_videos {valid_videos.shape}, valid_labels {valid_labels.shape}')
# print(f'test_videos {test_videos.shape}, test_labels {test_labels.shape}')

import torch
from torch.utils.data import DataLoader, TensorDataset, RandomSampler

def prepare_dataloader(
    videos: np.ndarray,
    labels: np.ndarray,
    loader_type: str = "train",
    batch_size: int = 32,  # Use the value of BATCH_SIZE if it's defined elsewhere
):
    """Utility function to prepare the dataloader."""
    # Convert numpy arrays to PyTorch tensors
    videos_tensor = torch.tensor(videos, dtype=torch.float32).clone().detach()  # Detach to avoid gradient tracking
    labels_tensor = torch.tensor(labels, dtype=torch.long).clone().detach()  # Detach labels similarly

    # Create TensorDataset
    dataset = TensorDataset(videos_tensor, labels_tensor)

    # If train data, use RandomSampler for shuffling
    if loader_type == "train":
        sampler = RandomSampler(dataset)
        shuffle = False  # Do not use shuffle because RandomSampler handles it
    else:
        sampler = None
        shuffle = False  # No shuffling for validation or test

    # Create DataLoader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,  # Use sampler for random sampling if it's train data
        shuffle=shuffle,  # Do not shuffle if using sampler
        num_workers=4,  # You can adjust this based on your system
    )

    return dataloader




trainloader = prepare_dataloader(train_videos, train_labels, "train")
print("done with trainloader")
validloader = prepare_dataloader(valid_videos, valid_labels, "valid")
print("done with validloader")
testloader = prepare_dataloader(test_videos, test_labels, "test")
print("done with testloader")


train_videos torch.Size([585, 1, 24, 89, 180]), train_labels torch.Size([585])
valid_videos torch.Size([14, 1, 24, 89, 180]), valid_labels torch.Size([14])
test_videos torch.Size([133, 1, 24, 89, 180]), test_labels torch.Size([133])
done with trainloader
done with validloader
done with testloader


/var/folders/56/lnrgwj295yq6l00hqdbfzfmr0000gn/T/ipykernel_7095/1000969914.py:98: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  videos_tensor = torch.tensor(videos, dtype=torch.float32).clone().detach()  # Detach to avoid gradient tracking
/var/folders/56/lnrgwj295yq6l00hqdbfzfmr0000gn/T/ipykernel_7095/1000969914.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_tensor = torch.tensor(labels, dtype=torch.long).clone().detach()  # Detach labels similarly


In [ ]:
print(vit_3d.run_experiment)  # Check if it's referencing your function
model, history = vit_3d.run_experiment(trainloader, testloader, validloader)

<function run_experiment at 0x158ddf9d0>
image size (89, 180) patch size (8, 8) frames 24 frame_patch_size 8
Padding width to 184 to make it divisible by 8
Padding height to 96 to make it divisible by 8
Number of Patches 828 Patch dim 512
Epoch [1/10], Loss: 28.2940, Accuracy: 0.2547
Validation Accuracy: 0.1880, Validation Loss: 7.040199160575867
Epoch [2/10], Loss: 27.3453, Accuracy: 0.2462
Validation Accuracy: 0.2481, Validation Loss: 6.931374549865723


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Constants
DATA_PATH = "data/cluster_1_sst.npz"
BATCH_SIZE = 32

class VideoDataset(Dataset):
    """Dataset for loading video data."""
    
    def __init__(self, videos, labels):
        self.videos = torch.tensor(videos, dtype=torch.float32)  # Convert to float32 for efficiency
        self.labels = torch.tensor(labels, dtype=torch.long)  # Assuming classification task
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]

def download_and_prepare_dataset(data_path):
    """Loads dataset from .npz file and returns PyTorch DataLoader objects."""
    with np.load(data_path, allow_pickle=True) as data:
        # Load and clean data
        train_videos = np.nan_to_num(data["train_images"])
        valid_videos = np.nan_to_num(data["val_images"])
        test_videos = np.nan_to_num(data["test_images"])

        train_labels = data["train_labels"].astype(np.int64)
        valid_labels = data["val_labels"].astype(np.int64)
        test_labels = data["test_labels"].astype(np.int64)

    return train_videos, train_labels, valid_videos, valid_labels, test_videos, test_labels

def prepare_dataloader(videos, labels, batch_size=BATCH_SIZE, shuffle=True):
    """Creates a PyTorch DataLoader."""
    dataset = VideoDataset(videos, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4, pin_memory=True)

# Usage
train_videos, train_labels, valid_videos, valid_labels, test_videos, test_labels = download_and_prepare_dataset(DATA_PATH)

print(f'train_videos {train_videos.shape}, train_labels {train_labels.shape}')
print(f'valid_videos {valid_videos.shape}, valid_labels {valid_labels.shape}')
print(f'test_videos {test_videos.shape}, test_labels {test_labels.shape}')

train_loader = prepare_dataloader(train_videos, train_labels)
print("done with trainloader")
valid_loader = prepare_dataloader(valid_videos, valid_labels, shuffle=False)
print("done with validloader")
test_loader = prepare_dataloader(test_videos, test_labels, shuffle=False)
print("done with testloader")

train_videos (585, 24, 89, 180), train_labels (585,)
valid_videos (14, 24, 89, 180), valid_labels (14,)
test_videos (133, 24, 89, 180), test_labels (133,)
done with trainloader
done with validloader
done with testloader


In [ ]:
print(vivit.run_experiment)  # Check if it's referencing your function
model, history = vivit.run_experiment(trainloader, testloader, validloader)

<function run_experiment at 0x35490df70>


NameError: name 'trainloader' is not defined